In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [4]:
with open('dracula.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '£', 'à', 'á', 'â', 'æ', 'è', 'é', 'ë', 'ï', 'ô', 'ö', '—', '‘', '’', '“', '”']
96


In [5]:
string_to_int= { ch:i for i,ch in enumerate(chars) }
int_to_string= { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ' '.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text))

print(data[:200])


tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 26, 40, 23, 25,
        43, 34, 23,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1, 51, 53, 76, 51,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1, 24, 69, 52, 64,  1, 41, 71, 66, 62, 56, 69,  0,  0,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 49, 31, 63, 63, 72, 70, 71, 69, 52, 71, 60, 66, 65, 20,
         1, 54, 66, 63, 66, 67, 59, 66, 65, 50,  0,  0,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1])


In [6]:
n = int(.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
   # print(ix)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1:i + block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x,y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 1, 54, 69, 56, 52, 71, 72, 69],
        [ 1, 74, 56,  1, 55, 56, 54, 60],
        [69,  9,  0, 30, 56,  1, 60, 70],
        [ 1, 65, 66, 71, 60, 54, 56,  9]], device='cuda:0')
targets:
tensor([[54, 69, 56, 52, 71, 72, 69, 56],
        [74, 56,  1, 55, 56, 54, 60, 55],
        [ 9,  0, 30, 56,  1, 60, 70,  1],
        [65, 66, 71, 60, 54, 56,  9,  1]], device='cuda:0')


In [7]:


x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target =y[t]
    print("when input is", context, "target is" , target)

when input is tensor([1]) target is tensor(1)
when input is tensor([1, 1]) target is tensor(1)
when input is tensor([1, 1, 1]) target is tensor(1)
when input is tensor([1, 1, 1, 1]) target is tensor(1)
when input is tensor([1, 1, 1, 1, 1]) target is tensor(1)
when input is tensor([1, 1, 1, 1, 1, 1]) target is tensor(1)
when input is tensor([1, 1, 1, 1, 1, 1, 1]) target is tensor(1)
when input is tensor([1, 1, 1, 1, 1, 1, 1, 1]) target is tensor(1)


In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
        

In [9]:
class DraculaLLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # nn.Embedding(vocab_size, vocab_size) is basically a lookup table

    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        #index is (B,T) array of indices in the current context
        for _ in range (max_new_tokens):
            # get predictions
            logits, loss = self.forward(index)
            # focus only on the last step
            logits = logits[:, -1, :] # becomes (B,C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (B,C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = DraculaLLM(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



            


 ‘ e ] s l j a w â , h T ” ‘ 3 S ” 7 ” X 8 S £ 8 9 b “ æ ) C o Z c ? D X b F c é k m { N { è u ô ï ’ ) f 
 K q 7 Q E á 2 ’ Z 4 3 K — { [ e s , P M Y á ’ , L g 1 f X é i ’ K ] “ i v V â p L 
 & è * o r A T 1 4 a w u ’ a * y ö ’ : 9 _ è ; o 0 à l Q A n _ ô I P à á D è B ( à 8 M b   1 R J p Q Z “ { B v K f z Z P ) _ ï Q N U G á ‘ ï b 0 i , h 0 * V Q ï 5 ; ô e f ” B V y z : l B }   L M — i ’ 4 8 Z — S ] ï J h U o C N y £ { 3 6 ’ o — ë h 7 & Z [ â ë s T ï c ’ é ] C q ë & ] M 5 ( 2 g d R C ” ( c R j K y ‘ 7 c   { ? L ‘ O O ë , o f ï 4 E ’ g A   — ö m C Y D c 1 £ è 2 O G j n h * w â v : O u _   C   y i 7 O r ? S g 
 á 4 h 7 ; I & 9 L g L ; H ö £ & J ’ U s j c : F ] 4 
 ‘ z ! 1 * o 0 P W à g ô 7 H e   ] ï - ö r A K 4 c W ] m 6 l , f z á 9 l i V _ U { S L “ 
 A B [ n ‘ à ô K 9 ] ( v 2 ‘ _ j J r à 
 & ] V N 0 _ d ” Z v . J v c ö R ‘ r , S é è D ‘ 1 C N c 1 t ” d Q & j f â r £ á I ö ï a R ö é 
 k f l Q u T 0 A W q ( V . t Z 8 g e E } j Q n T 3 Y à J ; Y _ U X ” ; Z è ; . W W , Y £ “ è ! T M c ? 

In [10]:
# creating a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}") #the f is for the decimal spot
        
    # sample a batch of data
    xb, yb =  get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 5.002, val loss: 4.984
step: 250, train loss: 4.912, val loss: 4.919
step: 500, train loss: 4.835, val loss: 4.848
step: 750, train loss: 4.797, val loss: 4.796
4.788662433624268


In [11]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


 b t q I 9 g ï E O s & n . c . c A c v [ X Y F ] . ( v A j c ë 5 q h r S C E X J o B z n   £ è 
 m 2 Q Z M J — S k z 7 Z 7 x l R é R a a “ t T ô ! j V V J [ à “ P 8 V ’ ) . o k q H D i j _ é ] l { (   N N ( e R é â D . 1 H J , ô H ë p Z 4 G ô s Q D Z ë 9 1 * ! F ’ d , ö ( ’ u K L R n R w Q w R 4 O G 
 
 ‘ * o 1 ; e è A   ’ i T ” ; £ £ w i æ K 8 i 4 8 v m 2 0 X æ f K & O — i N a k i t : J ; N á k { i } a “ 
 S M “   o r b E I X 8 o p ” Z k O i ’ 8 B U G 2 ” B â ( 3 , Y *   w ” M á g V N 4 y ( m K — 8 ( ï è w 5 ; ! . P ’ Z ] Z ë , l O O ï . t ] L ? [ X * s h W j C M ë i 4 ( H ô I : F a “ 7 K P è { V _ W ö 6 { 6 M 0 [ 4 A   “ x W L a “ 2 ö I 5 4 A j £ } ] 6 á ; P J v 0 L d d S S ö “ à s , E f z á V â p _ _ k O i u t M Y & z ‘ c X 2 N c j 9 2 l 5 o j J ö p “ [ q Y e e — : I q Y 5 ” ? L S ” E t q ( . P e X : ö s m N A B { Y ” ’ ô t 
 P { c é i f Y ” B 3 è 6 C à M D ) 
 o ( [ f M C X æ ) ë v T j u è ë Z   : F [ ” 7 _ ; w L o M ë ’ 8 r : ô I Y T M [ à _ d W d z x k V u P 8 2 ô T ’ ) u P U _ 